In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. **Install Ollama** from [https://ollama.com/download](https://ollama.com/download)
   - **macOS**: download the `.pkg` and install it
   - **Windows**: download the `.msi` and install it
   - **Linux**: run
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model** in a terminal:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model and starts it locally with a chat-like interface.

3. **Check that the local server is running**:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}
   ```

If you want to use Ollama from Python, install the client with:

```bash
pip install ollama
```

and then call it like this:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```


In [5]:
# make a typo in the question to test the RAG model's ability to handle misspellings
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see anything in the course FAQ about running **Ollama** locally.

The FAQ only says you can run the **course locally** instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and the other needed tools, and that you should document your setup and keep it reproducible.

If you meant something else by “Olama,” let me know and I’ll check the relevant FAQ context.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Yes—usually you can join, but it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out. Please send:\n- the course name\n- where it’s offered\n- whether it’s live or self-paced\n- the start date or current date in the course\n\nIf you meant to ask the course instructor or organizer, you could say:\n\n> Hi, I just discovered this course. Is it still possible for me to join?\n\nIf you want, I can also help you write a more formal or more casual version.'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course enrollment late join"}', call_id='call_DBKNY5yE7ypvOMDCO3iyecTe', name='search', type='function_call', id='fc_0b4d2d0988f2004f006a64de28d79481958547ed7bfdaa7e89', caller=None, namespace=None, status='completed')]

In [12]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course enrollment late join"}', call_id='call_DBKNY5yE7ypvOMDCO3iyecTe', name='search', type='function_call', id='fc_0b4d2d0988f2004f006a64de28d79481958547ed7bfdaa7e89', caller=None, namespace=None, status='completed')

In [13]:
call.arguments

'{"query":"Can I join the course late discovered the course enrollment late join"}'

In [14]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course late discovered the course enrollment late join'}

In [15]:
call.name

'search'

In [16]:
results = search(**args)

In [17]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch

In [18]:
result_json = json.dumps(results, indent=2)

In [19]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [20]:
messages.append(call)

In [21]:
messages.append(function_call_output)

In [22]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course enrollment late join"}', call_id='call_DBKNY5yE7ypvOMDCO3iyecTe', name='search', type='function_call', id='fc_0b4d2d0988f2004f006a64de28d79481958547ed7bfdaa7e89', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_DBKNY5yE7ypvOMDCO3iyecTe',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workf

In [23]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [24]:
print(response.output_text)

Yes — you can still join and start learning.

If you want a certificate, though, you’ll need to submit your project while the course is still accepting submissions.


In [25]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(774, 37)

In [26]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [27]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [29]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [30]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration FAQ"}', call_id='call_XWKk3yJPygCbG8ycenh8ppQd', name='search', type='function_call', id='fc_09b33b6f4e1b6e82006a64de2bdaf881938e30fd922c26436e', caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment add drop late join registration FAQ"}', call_id='call_z7nothtlhcziDfuxrbu9jsNJ', name='search', type='function_call', id='fc_09b33b6f4e1b6e82006a64de2bdb0c81938b57af13983ef42c', caller=None, namespace=None, status='completed')]

In [31]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
function_call: search {"query":"course enrollment add drop late join registration FAQ"}


In [32]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration FAQ"}', call_id='call_XWKk3yJPygCbG8ycenh8ppQd', name='search', type='function_call', id='fc_09b33b6f4e1b6e82006a64de2bdaf881938e30fd922c26436e', caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment add drop late join registration FAQ

In [33]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment register late course FAQ"}
function_call: search {"query":"course access enrollment new student join after start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

One important caveat: if you want a certificate, you need to submit your project while submissions are still being accepted. You can also follow the course in a self-paced way, but the certificate is only available if you finish with a live cohort.

If you want, I can also help you with:
- how to start the course,
- whether you can still get a certificate,
- or the weekly workflow and deadlines.


### Agent loop

In [ ]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [36]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course enroll discovered course can I join late registration enrollment"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

One important note: if you want a certificate, you need to submit your project while submissions are still being accepted in the live cohort.

If you'd like, I can also help you figure out how to start from the current point in the course. Any other areas you want to explore?


In [37]:
result

"Yes — you can still join the course.\n\nOne important note: if you want a certificate, you need to submit your project while submissions are still being accepted in the live cohort.\n\nIf you'd like, I can also help you figure out how to start from the current point in the course. Any other areas you want to explore?"

In [38]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit chess opening Queen's Gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit course FAQ"}
iteration #3...
ASSISTANT:
If you meant the **Queen’s Gambit** in chess, I’m sorry, but I can’t answer that from the course FAQ database.

If you meant something related to the course, feel free to rephrase your question and I’ll look it up. Is there another area you want to explore?
